In [ ]:
#Logistic regression using scikit on Phishing URL Dataset

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


def main():
    df = pd.read_csv("PhiUSIIL_Phishing_URL_Dataset.csv")

    # Binary target: phishing vs legitimate
    target_col = "label"

    # Use only numeric columns as features for a simple scikit-learn baseline
    feature_cols = [
        c for c in df.select_dtypes(include=["number"]).columns.tolist()
        if c != target_col
    ]

    X = df[feature_cols]
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)

    model = LogisticRegression(
        max_iter=2000,
        random_state=42,
        class_weight="balanced",
    )
    model.fit(X_train_imp, y_train)

    y_pred = model.predict(X_test_imp)

    F1_score = classification_report(y_test, y_pred, output_dict=True)['weighted avg']['f1-score']

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", F1_score)
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

    # Show coefficients for the strongest positive/negative predictors
    coef = model.coef_[0]
    coef_rank = sorted(
        zip(feature_cols, coef),
        key=lambda x: abs(x[1]),
        reverse=True,
    )[:10]

    print("\nTop 10 feature weights:")
    for feature, weight in coef_rank:
        print(f"{feature}: {weight:.6f}")


if __name__ == "__main__":
    main()


Accuracy: 0.999978795139846
F1 Score: 0.9999787950738163

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159


Confusion Matrix:
 [[20188     1]
 [    0 26970]]

Top 10 feature weights:
IsHTTPS: 7.494532
DomainLength: 5.206263
URLLength: -4.938136
HasDescription: 2.371741
HasCopyrightInfo: 2.139158
NoOfOtherSpecialCharsInURL: -2.075865
HasSocialNet: 2.017446
CharContinuationRate: -1.987859
NoOfDegitsInURL: -1.412312
HasFavicon: 1.359249


c:\Users\JL\Desktop\Student projects\Cryptonite\Cryptonite\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
